In [1]:
!pip install -q chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 68.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 58.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 60.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 6.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently 

In [2]:
import chromadb

client = chromadb.Client()

library = client.create_collection(name="library_demo")

print("Collection:", library.name)
print("Initial records:", library.count())

Collection: library_demo
Initial records: 0


In [3]:
library.add(
    documents=[
        "Python books are useful for learning programming and automation.",
        "The Himalayas are popular for trekking and mountain adventures.",
        "Italian cuisine includes pasta, pizza, and many regional dishes.",
        "Data structures organize information efficiently for algorithms.",
        "Photography involves composition, lighting, and camera settings."
    ],
    metadatas=[
        {"topic": "programming"},
        {"topic": "travel"},
        {"topic": "food"},
        {"topic": "computer_science"},
        {"topic": "photography"}
    ],
    ids=["book101", "book102", "book103", "book104", "book105"]
)

print("Records added:", library.count())

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:02<00:00, 40.3MiB/s]


Records added: 5


In [4]:
preview = library.peek(limit=3)

for item_id, text in zip(preview["ids"], preview["documents"]):
    print(f"{item_id} -> {text}")

book101 -> Python books are useful for learning programming and automation.
book102 -> The Himalayas are popular for trekking and mountain adventures.
book103 -> Italian cuisine includes pasta, pizza, and many regional dishes.


In [5]:
selected = library.get(ids=["book101", "book104"])

for item_id, text, meta in zip(
    selected["ids"], selected["documents"], selected["metadatas"]
):
    print(f"{item_id} | {meta} | {text}")

book101 | {'topic': 'programming'} | Python books are useful for learning programming and automation.
book104 | {'topic': 'computer_science'} | Data structures organize information efficiently for algorithms.


In [6]:
results = library.query(
    query_texts=["learning algorithms and programming"],
    n_results=3
)

for item_id, text, distance in zip(
    results["ids"][0],
    results["documents"][0],
    results["distances"][0]
):
    print(f"{item_id} | distance={distance:.4f} | {text}")

book104 | distance=1.0882 | Data structures organize information efficiently for algorithms.
book101 | distance=1.2746 | Python books are useful for learning programming and automation.
book105 | distance=1.8189 | Photography involves composition, lighting, and camera settings.


In [7]:
results = library.query(
    query_texts=["places and outdoor activities"],
    n_results=3,
    where={"topic": "travel"}
)

for item_id, text, distance in zip(
    results["ids"][0],
    results["documents"][0],
    results["distances"][0]
):
    print(f"{item_id} | distance={distance:.4f} | {text}")

book102 | distance=1.3353 | The Himalayas are popular for trekking and mountain adventures.


In [8]:
library.update(
    ids=["book103"],
    documents=["Italian cuisine includes pasta, pizza, risotto, and regional desserts."],
    metadatas=[{"topic": "food", "updated": True}]
)

updated = library.get(ids=["book103"])
print("Document:", updated["documents"])
print("Metadata:", updated["metadatas"])

Document: ['Italian cuisine includes pasta, pizza, risotto, and regional desserts.']
Metadata: [{'topic': 'food', 'updated': True}]


In [9]:
library.upsert(
    documents=[
        "Data structures such as trees and graphs are important in computer science.",
        "Watercolor painting uses pigments mixed with water to create transparent layers."
    ],
    metadatas=[
        {"topic": "computer_science", "version": 2},
        {"topic": "art"}
    ],
    ids=["book104", "book106"]
)

print("Count after upsert:", library.count())
print(library.get(ids=["book104", "book106"])["documents"])

Count after upsert: 6
['Data structures such as trees and graphs are important in computer science.', 'Watercolor painting uses pigments mixed with water to create transparent layers.']


In [10]:
library.delete(ids=["book102"])

print("Count after deleting book102:", library.count())
print("Remaining IDs:", library.get()["ids"])

Count after deleting book102: 5
Remaining IDs: ['book101', 'book103', 'book104', 'book105', 'book106']


In [11]:
library.delete(where={"topic": "computer_science"})

print("Count after deleting computer-science records:", library.count())
print("Remaining IDs:", library.get()["ids"])

Count after deleting computer-science records: 4
Remaining IDs: ['book101', 'book103', 'book105', 'book106']


In [12]:
final_state = library.get()

print("Final record count:", library.count())
for item_id, text, meta in zip(
    final_state["ids"],
    final_state["documents"],
    final_state["metadatas"]
):
    print(f"{item_id} | {meta} | {text}")

Final record count: 4
book101 | {'topic': 'programming'} | Python books are useful for learning programming and automation.
book103 | {'updated': True, 'topic': 'food'} | Italian cuisine includes pasta, pizza, risotto, and regional desserts.
book105 | {'topic': 'photography'} | Photography involves composition, lighting, and camera settings.
book106 | {'topic': 'art'} | Watercolor painting uses pigments mixed with water to create transparent layers.
